# Fake News Detection — Extended Pipeline

Extends `main.ipynb` with:
1. Vectorizer experiments (BoW & TF-IDF, multiple parameter sets)
2. Random Forest classifier
3. Hyperparameter tuning with `RandomizedSearchCV`
4. Full model comparison table
5. Prediction on `testing_data.csv` (replaces label `2` with 0 or 1)

In [1]:
import re
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd

import nltk
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords, wordnet
from nltk.stem import WordNetLemmatizer

from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.preprocessing import Normalizer
from sklearn.model_selection import train_test_split, RandomizedSearchCV, StratifiedKFold
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix

for pkg in ['punkt','punkt_tab','stopwords','wordnet',
            'averaged_perceptron_tagger','averaged_perceptron_tagger_eng']:
    nltk.download(pkg, quiet=True)

RANDOM_STATE = 42
print('All imports OK')

All imports OK


## Load Pre-processed Training Data

We load `cleaned_training_data.csv` produced by the original `main.ipynb`. It already contains `text_lemmatized` — the fully cleaned, stop-word-free, lemmatized column — so we skip re-processing the training set.

In [2]:
df = pd.read_csv('dataset/cleaned_training_data.csv')
df = df[df['text_lemmatized'].notna()]
df = df[df['text_lemmatized'].str.strip() != '']

print('Shape:', df.shape)
print('Label distribution:')
print(df['label'].value_counts())
df.head(3)

Shape: (32203, 5)
Label distribution:
label
1    16181
0    16022
Name: count, dtype: int64


,label,text,text_clean,text_no_stopwords,text_lemmatized
0,0,donald trump sends out embarrassing new year‚s...,donald trump sends out embarrassing new year e...,donald trump sends embarrassing new year eve m...,donald trump sends embarrassing new year eve m...
1,0,drunk bragging trump staffer started russian c...,drunk bragging trump staffer started russian c...,drunk bragging trump staffer started russian c...,drunk bragging trump staffer started russian c...
2,0,sheriff david clarke becomes an internet joke ...,sheriff david clarke becomes an internet joke ...,sheriff david clarke becomes internet joke thr...,sheriff david clarke becomes internet joke thr...


## Train / Validation Split

In [3]:
X = df['text_lemmatized']
y = df['label']

X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)

print(f'Train : {X_train.shape[0]} rows')
print(f'Val   : {X_val.shape[0]} rows')

Train : 25762 rows
Val   : 6441 rows


## Custom Tokenizer

A regex-based tokenizer that keeps only alphanumeric tokens of 2+ characters. Used as one of the vectorizer parameter variants.

In [4]:
def custom_tokenizer(text):
    """Keep alphanumeric tokens with 2+ chars, lowercased."""
    return re.findall(r'\b[a-z0-9]{2,}\b', text.lower())

# Sanity check
print(custom_tokenizer('Trump sent an embarrassing tweet in 2020!'))

['trump', 'sent', 'an', 'embarrassing', 'tweet', 'in', '2020']


## Step 1 — Vectorizer Experiments

We define **7 configurations** — 3 Bag-of-Words and 4 TF-IDF — varying:

| Parameter | Values tried |
|---|---|
| `ngram_range` | (1,1) unigrams · (1,2) bigrams |
| `max_df` | 1.0 (no cap) · 0.95 · 0.90 |
| `min_df` | 1 · 2 · 3 |
| `max_features` | None · 10 000 · 15 000 · 20 000 |
| `tokenizer` | default · custom regex |
| `sublinear_tf` | False · True (TF-IDF only) |

Each config is evaluated with a **baseline Random Forest** (100 trees, entropy criterion) so the vectorizer impact is isolated.

In [5]:
vec_configs = [
    # ── Bag of Words ─────────────────────────────────────────────────────────
    {
        'name'  : 'BoW_unigram_default',
        'type'  : 'count',
        'params': dict(ngram_range=(1,1), max_df=1.0, min_df=1, max_features=None)
    },
    {
        'name'  : 'BoW_bigram_10k',
        'type'  : 'count',
        'params': dict(ngram_range=(1,2), max_df=0.95, min_df=2, max_features=10_000)
    },
    {
        'name'  : 'BoW_unigram_custom_tok_20k',
        'type'  : 'count',
        'params': dict(ngram_range=(1,1), max_df=0.90, min_df=3,
                       max_features=20_000, tokenizer=custom_tokenizer)
    },
    # ── TF-IDF ───────────────────────────────────────────────────────────────
    {
        'name'  : 'TFIDF_unigram_default',
        'type'  : 'tfidf',
        'params': dict(ngram_range=(1,1), max_df=1.0, min_df=1, max_features=None)
    },
    {
        'name'  : 'TFIDF_bigram_15k',
        'type'  : 'tfidf',
        'params': dict(ngram_range=(1,2), max_df=0.95, min_df=2, max_features=15_000)
    },
    {
        'name'  : 'TFIDF_unigram_custom_tok_20k',
        'type'  : 'tfidf',
        'params': dict(ngram_range=(1,1), max_df=0.90, min_df=3,
                       max_features=20_000, tokenizer=custom_tokenizer)
    },
    {
        'name'  : 'TFIDF_bigram_sublinear_20k',
        'type'  : 'tfidf',
        'params': dict(ngram_range=(1,2), max_df=0.90, min_df=2,
                       max_features=20_000, sublinear_tf=True)
    },
]

In [6]:
normalizer_l2 = Normalizer(norm='l2')
vec_results   = []

for cfg in vec_configs:
    # Build vectorizer
    if cfg['type'] == 'count':
        vec = CountVectorizer(**cfg['params'])
    else:
        vec = TfidfVectorizer(**cfg['params'])

    X_tr = vec.fit_transform(X_train)
    X_vl = vec.transform(X_val)

    # L2-normalise BoW (TF-IDF is already unit-normalised)
    if cfg['type'] == 'count':
        X_tr = normalizer_l2.fit_transform(X_tr)
        X_vl = normalizer_l2.transform(X_vl)

    # Baseline RF
    rf = RandomForestClassifier(
        n_estimators=100, criterion='entropy',
        random_state=RANDOM_STATE, n_jobs=-1
    )
    rf.fit(X_tr, y_train)
    y_pred = rf.predict(X_vl)

    acc   = accuracy_score(y_val, y_pred)
    f1    = f1_score(y_val, y_pred, average='weighted')
    vocab = len(vec.vocabulary_)

    vec_results.append({
        'config'     : cfg['name'],
        'vec_type'   : cfg['type'],
        'vocab_size' : vocab,
        'accuracy'   : round(acc, 4),
        'f1_weighted': round(f1, 4),
        'model_tag'  : 'baseline_RF'
    })
    print(f"[{cfg['name']}]  acc={acc:.4f}  f1={f1:.4f}  vocab={vocab:,}")

vec_df = pd.DataFrame(vec_results).sort_values('f1_weighted', ascending=False)
print('\n── Vectorizer Experiment Results (sorted by F1) ──')
print(vec_df.to_string(index=False))

[BoW_unigram_default]  acc=0.9152  f1=0.9152  vocab=14,968
[BoW_bigram_10k]  acc=0.9079  f1=0.9079  vocab=10,000
[BoW_unigram_custom_tok_20k]  acc=0.9073  f1=0.9073  vocab=7,196
[TFIDF_unigram_default]  acc=0.9112  f1=0.9112  vocab=14,968
[TFIDF_bigram_15k]  acc=0.9084  f1=0.9084  vocab=15,000
[TFIDF_unigram_custom_tok_20k]  acc=0.9073  f1=0.9073  vocab=7,196
[TFIDF_bigram_sublinear_20k]  acc=0.9050  f1=0.9049  vocab=20,000

── Vectorizer Experiment Results (sorted by F1) ──
                      config vec_type  vocab_size  accuracy  f1_weighted   model_tag
         BoW_unigram_default    count       14968    0.9152       0.9152 baseline_RF
       TFIDF_unigram_default    tfidf       14968    0.9112       0.9112 baseline_RF
            TFIDF_bigram_15k    tfidf       15000    0.9084       0.9084 baseline_RF
              BoW_bigram_10k    count       10000    0.9079       0.9079 baseline_RF
  BoW_unigram_custom_tok_20k    count        7196    0.9073       0.9073 baseline_RF
TFIDF_unig

### Select Best Vectorizer Config

Pick the configuration with the highest weighted F1 and rebuild its train/val matrices for tuning.

In [7]:
best_vec_name = vec_df.iloc[0]['config']
best_cfg      = next(c for c in vec_configs if c['name'] == best_vec_name)
print('Best vectorizer config:', best_vec_name)

if best_cfg['type'] == 'count':
    best_vec = CountVectorizer(**best_cfg['params'])
else:
    best_vec = TfidfVectorizer(**best_cfg['params'])

X_train_best = best_vec.fit_transform(X_train)
X_val_best   = best_vec.transform(X_val)

if best_cfg['type'] == 'count':
    norm_final   = Normalizer(norm='l2')
    X_train_best = norm_final.fit_transform(X_train_best)
    X_val_best   = norm_final.transform(X_val_best)
else:
    norm_final = None   # TF-IDF already normalised

print('Train matrix:', X_train_best.shape)
print('Val   matrix:', X_val_best.shape)

Best vectorizer config: BoW_unigram_default
Train matrix: (25762, 14968)
Val   matrix: (6441, 14968)


## Steps 2 & 3 — Random Forest + Hyperparameter Tuning

`RandomizedSearchCV` with 5-fold stratified CV searches over:

| Parameter | Search space |
|---|---|
| `n_estimators` | 100, 200, 300, 500 |
| `max_depth` | None, 20, 40, 60, 80 |
| `min_samples_split` | 2, 5, 10 |
| `min_samples_leaf` | 1, 2, 4 |
| `max_features` | 'sqrt', 'log2' |
| `criterion` | 'gini', 'entropy' |

Scoring metric: **weighted F1**.

In [8]:
param_dist = {
    'n_estimators'     : [100, 200, 300, 500],
    'max_depth'        : [None, 20, 40, 60, 80],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf' : [1, 2, 4],
    'max_features'     : ['sqrt', 'log2'],
    'criterion'        : ['gini', 'entropy'],
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

rscv = RandomizedSearchCV(
    estimator           = RandomForestClassifier(random_state=RANDOM_STATE, n_jobs=-1),
    param_distributions = param_dist,
    n_iter              = 20,
    scoring             = 'f1_weighted',
    cv                  = cv,
    random_state        = RANDOM_STATE,
    n_jobs              = -1,
    verbose             = 1,
    refit               = True
)

print('Running RandomizedSearchCV (20 iterations × 5-fold CV) …')
rscv.fit(X_train_best, y_train)

print('\nBest parameters :', rscv.best_params_)
print('Best CV F1      :', round(rscv.best_score_, 4))

Running RandomizedSearchCV (20 iterations × 5-fold CV) …
Fitting 5 folds for each of 20 candidates, totalling 100 fits

Best parameters : {'n_estimators': 500, 'min_samples_split': 2, 'min_samples_leaf': 1, 'max_features': 'log2', 'max_depth': None, 'criterion': 'gini'}
Best CV F1      : 0.9351


In [9]:
best_rf       = rscv.best_estimator_
y_pred_tuned  = best_rf.predict(X_val_best)

acc_tuned = accuracy_score(y_val, y_pred_tuned)
f1_tuned  = f1_score(y_val, y_pred_tuned, average='weighted')

print(f'Tuned RF — Validation Accuracy  : {acc_tuned:.4f}')
print(f'Tuned RF — Validation F1 (wt.)  : {f1_tuned:.4f}')
print()
print('Classification Report:')
print(classification_report(y_val, y_pred_tuned))
print('Confusion Matrix:')
print(confusion_matrix(y_val, y_pred_tuned))

Tuned RF — Validation Accuracy  : 0.9359
Tuned RF — Validation F1 (wt.)  : 0.9359

Classification Report:
              precision    recall  f1-score   support

           0       0.95      0.92      0.93      3205
           1       0.92      0.95      0.94      3236

    accuracy                           0.94      6441
   macro avg       0.94      0.94      0.94      6441
weighted avg       0.94      0.94      0.94      6441

Confusion Matrix:
[[2942  263]
 [ 150 3086]]


## Step 4 — Model Comparison

All vectorizer × baseline-RF runs **plus** the tuned RF, ranked by weighted F1.

In [10]:
all_results = vec_df.copy()

tuned_row = pd.DataFrame([{
    'config'     : f'{best_vec_name}  +  Tuned RF',
    'vec_type'   : best_cfg['type'],
    'vocab_size' : len(best_vec.vocabulary_),
    'accuracy'   : round(acc_tuned, 4),
    'f1_weighted': round(f1_tuned, 4),
    'model_tag'  : 'tuned_RF'
}])

all_results = (
    pd.concat([all_results, tuned_row], ignore_index=True)
    .sort_values('f1_weighted', ascending=False)
    .reset_index(drop=True)
)

print('══ Full Model Comparison (sorted by F1) ══')
print(all_results[['config','model_tag','accuracy','f1_weighted']].to_string(index=False))

══ Full Model Comparison (sorted by F1) ══
                          config   model_tag  accuracy  f1_weighted
BoW_unigram_default  +  Tuned RF    tuned_RF    0.9359       0.9359
             BoW_unigram_default baseline_RF    0.9152       0.9152
           TFIDF_unigram_default baseline_RF    0.9112       0.9112
                TFIDF_bigram_15k baseline_RF    0.9084       0.9084
                  BoW_bigram_10k baseline_RF    0.9079       0.9079
      BoW_unigram_custom_tok_20k baseline_RF    0.9073       0.9073
    TFIDF_unigram_custom_tok_20k baseline_RF    0.9073       0.9073
      TFIDF_bigram_sublinear_20k baseline_RF    0.9050       0.9049


In [11]:
winner = all_results.iloc[0]
print(f"  BEST MODEL : {winner['config']}")
print(f"   model_tag  : {winner['model_tag']}")
print(f"   Accuracy   : {winner['accuracy']}")
print(f"   F1 (wt.)   : {winner['f1_weighted']}")

  BEST MODEL : BoW_unigram_default  +  Tuned RF
   model_tag  : tuned_RF
   Accuracy   : 0.9359
   F1 (wt.)   : 0.9359


## Step 5 — Predict on testing_data.csv

Pipeline:
1. Load the raw test file (tab-separated, label column contains `'2'`)
2. Re-apply the same text cleaning → stopword removal → lemmatization
3. Vectorize with the **fitted** `best_vec` (no re-fitting)
4. Predict with the best model
5. Replace the `2` placeholder with predicted 0/1 and save

In [12]:
test_raw = pd.read_csv(
    'dataset/testing_data.csv',
    sep='\t', header=None, names=['label', 'text']
)
print('Testing set shape:', test_raw.shape)
print('Label value counts:')
print(test_raw['label'].value_counts())

# Keep only rows where the label placeholder is '2'
# (a handful of rows have a BOM-prefixed label from the file encoding — treat as non-test)
test_df = test_raw.copy()
test_df['text'] = test_df['text'].fillna('')
test_df.head(3)

Testing set shape: (9984, 2)
Label value counts:
label
2     9982
﻿0       2
Name: count, dtype: int64


,label,text
0,2,copycat muslim terrorist arrested with assault...
1,2,wow! chicago protester caught on camera admits...
2,2,germany's fdp look to fill schaeuble's big shoes


In [16]:
def clean_text(text):
    text = re.sub(r"[^a-zA-Z0-9\s]", " ", str(text))
    text = re.sub(r"\s+[a-zA-Z]\s+", " ", text)
    text = re.sub(r"\^[a-zA-Z]\s+", " ", text)
    text = re.sub(r"\s+", " ", text)
    return text.lower().strip()

def remove_stopwords(text):
    stop_words = set(stopwords.words('english'))
    return ' '.join(w for w in word_tokenize(text) if w not in stop_words)

_lemmatizer = WordNetLemmatizer()

def get_wordnet_pos(tag):
    return {'J': wordnet.ADJ, 'V': wordnet.VERB, 'R': wordnet.ADV}.get(tag[0], wordnet.NOUN)

def lemmatize_text(text):
    tokens   = word_tokenize(text)
    pos_tags = nltk.pos_tag(tokens)
    return ' '.join(_lemmatizer.lemmatize(w, get_wordnet_pos(t)) for w, t in pos_tags)

print('Step 1/3 — Cleaning …')
test_df['text_clean'] = test_df['text'].apply(clean_text)

print('Step 2/3 — Removing stopwords …')
test_df['text_no_stopwords'] = test_df['text_clean'].apply(remove_stopwords)

print('Step 3/3 — Lemmatizing …')
test_df['text_lemmatized'] = test_df['text_no_stopwords'].apply(lemmatize_text)

print('Done.')
test_df[['text', 'text_lemmatized']].head(3)

Step 1/3 — Cleaning …
Step 2/3 — Removing stopwords …
Step 3/3 — Lemmatizing …
Done.


,text,text_lemmatized
0,copycat muslim terrorist arrested with assault...,copycat muslim terrorist arrest assault weapon
1,wow! chicago protester caught on camera admits...,wow chicago protester catch camera admits viol...
2,germany's fdp look to fill schaeuble's big shoes,germany fdp look fill schaeuble big shoe


In [17]:
# Vectorize using the FITTED vectorizer (transform only — never re-fit)
X_test_vec = best_vec.transform(test_df['text_lemmatized'])

if norm_final is not None:   # apply the same L2 normaliser if BoW was selected
    X_test_vec = norm_final.transform(X_test_vec)

print('Test matrix shape:', X_test_vec.shape)

Test matrix shape: (9984, 14968)


In [18]:
# Select the model that topped the comparison table
if winner['model_tag'] == 'tuned_RF':
    final_model = best_rf
    print('Using: Tuned RF (RandomizedSearchCV best estimator)')
else:
    # Re-train the best baseline config on the full training data for maximum coverage
    final_model = RandomForestClassifier(
        n_estimators=100, criterion='entropy',
        random_state=RANDOM_STATE, n_jobs=-1
    )
    final_model.fit(X_train_best, y_train)
    print(f'Using: Baseline RF with {best_vec_name}')

Using: Tuned RF (RandomizedSearchCV best estimator)


In [19]:
y_test_pred = final_model.predict(X_test_vec)

print('Prediction distribution:')
vals, counts = np.unique(y_test_pred, return_counts=True)
for v, c in zip(vals, counts):
    print(f'  label {v}: {c} rows')

Prediction distribution:
  label 0: 4755 rows
  label 1: 5229 rows


In [20]:
# Build the output: reload original file to preserve raw text exactly,
# then replace the label column with predictions.
output_df = pd.read_csv(
    'dataset/testing_data.csv',
    sep='\t', header=None, names=['label', 'text']
)

output_df['label'] = y_test_pred.astype(int)

output_path = 'dataset/testing_predictions.csv'
output_df.to_csv(output_path, sep='\t', header=False, index=False)

print(f'Saved → {output_path}')
print(f'Total predictions: {len(output_df)}')
print('\nFirst 10 rows of output:')
print(output_df.head(10).to_string(index=False))

Saved → dataset/testing_predictions.csv
Total predictions: 9984

First 10 rows of output:
 label                                                                                                          text
     0                                                        copycat muslim terrorist arrested with assault weapons
     0 wow! chicago protester caught on camera admits violent activity was pre-planned: ‚it‚s not gonna be peaceful‚
     1                                                              germany's fdp look to fill schaeuble's big shoes
     0                          mi school sends welcome back packet warning kids against wearing u.s. flag to school
     1                                  u.n. seeks 'massive' aid boost amid rohingya 'emergency within an emergency'
     0                          did oprah just leave ‚nasty‚ hillary wishing she wouldn‚t have endorsed her? [video]
     1                                     france's macron says his job not 'cool' cites ta

## Summary

| Step | What was done |
|---|---|
| **1 – Vectorizers** | 7 configs: 3 BoW + 4 TF-IDF varying n-grams, max/min_df, max_features, custom tokenizer, sublinear_tf |
| **2 – Classifier** | Random Forest (baseline: 100 trees, entropy) benchmarked on every config |
| **3 – Tuning** | `RandomizedSearchCV` 20 iter × 5-fold CV on best vectorizer |
| **4 – Comparison** | All runs ranked by weighted F1; best model auto-selected |
| **5 – Prediction** | Full preprocessing reapplied to test set; predictions saved to `dataset/testing_predictions.csv` |